# Weaviate Backend — Knowledge Graph + Vector Search

Medha 0.3.1 ships with `WeaviateBackend`: a storage backend backed by
[Weaviate](https://weaviate.io/), an open-source vector database that combines
semantic vector search with structured knowledge-graph filtering.

Weaviate is a strong choice when you need **advanced metadata filtering**
(e.g., tenant isolation, date ranges, tag filters) combined with vector search,
or when you want a managed cloud vector DB with multi-tenancy built in.

This notebook covers:
1. **Local mode** — self-hosted Weaviate via Docker
2. **Store & search** — full waterfall + `stats()`
3. **Cloud mode** — Weaviate Cloud Services (placeholder)
4. **Multi-collection** — per-tenant isolation
5. **Weaviate vs Qdrant** — when to choose which

**Requirements:**
```bash
pip install "medha-archai[weaviate,fastembed]"
```

## Prerequisites — Weaviate via Docker

```bash
docker run -d --name weaviate \
  -e AUTHENTICATION_ANONYMOUS_ACCESS_ENABLED=true \
  -e PERSISTENCE_DATA_PATH=/var/lib/weaviate \
  -p 8080:8080 -p 50051:50051 \
  cr.weaviate.io/semitechnologies/weaviate:1.25.0

# Verify
curl http://localhost:8080/v1/.well-known/ready
```

In [ ]:
import os
import time
import urllib.request

from medha import Medha, Settings
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

try:
    from medha import WeaviateBackend
    HAS_WEAVIATE = True
    print("weaviate-client package is installed")
except ImportError:
    HAS_WEAVIATE = False
    print("weaviate-client not found — install with: pip install medha-archai[weaviate]")

try:
    urllib.request.urlopen("http://localhost:8080/v1/.well-known/ready", timeout=2)
    WEAVIATE_UP = True
    print("Weaviate reachable at http://localhost:8080")
except Exception:
    WEAVIATE_UP = False
    print("Weaviate not reachable — cells requiring Weaviate will be skipped")

CAN_RUN = HAS_WEAVIATE and WEAVIATE_UP

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

## Cell 1: Local Mode — `weaviate_mode="local"`

In [ ]:
if not CAN_RUN:
    print("Skipping — Weaviate not available.")
else:
    settings = Settings(
        backend_type="weaviate",
        weaviate_mode="local",
        weaviate_host="localhost",
        weaviate_http_port=8080,
        weaviate_grpc_port=50051,
        weaviate_collection_prefix="Demo",
    )

    print(f"backend_type              : {settings.backend_type}")
    print(f"weaviate_mode             : {settings.weaviate_mode}")
    print(f"weaviate_collection_prefix: {settings.weaviate_collection_prefix}")

    async with Medha("wv_demo", embedder=embedder, settings=settings) as medha:
        print(f"\nBackend: {type(medha._backend).__name__}")

## Cell 2: Store 3 Entries + Search + `stats()`

In [ ]:
if not CAN_RUN:
    print("Skipping — Weaviate not available.")
else:
    settings = Settings(
        backend_type="weaviate",
        weaviate_mode="local",
        weaviate_collection_prefix="Demo",
    )

    pairs = [
        ("How many users are there?",   "SELECT COUNT(*) FROM users"),
        ("List active sessions",        "SELECT * FROM sessions WHERE active = true"),
        ("Monthly revenue summary",     "SELECT SUM(amount) FROM orders WHERE MONTH(created_at) = MONTH(NOW())"),
    ]

    async with Medha("wv_demo", embedder=embedder, settings=settings) as medha:
        for q, sql in pairs:
            await medha.store(q, sql)

        test_qs = [
            ("Total user count",                "semantic"),
            ("How many users are there?",        "exact/L1"),
            ("Something completely unrelated",  "miss"),
        ]

        print(f"  {'Question':<40s}  {'Strategy':<20s}  Conf")
        print("  " + "-" * 72)
        for q, note in test_qs:
            hit = await medha.search(q)
            conf = f"{hit.confidence:.4f}" if hit.confidence else "  n/a"
            print(f"  {q:<40s}  {hit.strategy.value:<20s}  {conf}  ({note})")

        stats = await medha.stats()
        print(f"\nStats: requests={stats.total_requests}  hit_rate={stats.hit_rate:.2%}")

## Cell 3: Cloud Mode — Weaviate Cloud Services (Placeholder)

This cell demonstrates the settings for Weaviate Cloud. It is **not executed**
— replace the placeholder values with your actual credentials.

In [ ]:
# ⚠️ Placeholder — do not execute without real credentials
settings_cloud = Settings(
    backend_type="weaviate",
    weaviate_mode="cloud",
    weaviate_cloud_url="https://my-cluster.weaviate.network",
    weaviate_api_key="my-weaviate-api-key",   # MEDHA_WEAVIATE_API_KEY
    weaviate_collection_prefix="Prod",
)

print(f"weaviate_mode     : {settings_cloud.weaviate_mode}")
print(f"weaviate_cloud_url: {settings_cloud.weaviate_cloud_url}")
print(f"api_key set       : {settings_cloud.weaviate_api_key is not None}")
print("(Not connecting — placeholder values)")

## Cell 4: Multi-Collection — Per-Tenant Isolation

In [ ]:
if not CAN_RUN:
    print("Skipping — Weaviate not available.")
else:
    settings = Settings(
        backend_type="weaviate",
        weaviate_mode="local",
        weaviate_collection_prefix="Demo",
    )

    # Two tenants share one Weaviate instance but use different collection names.
    # Weaviate collection names: DemoTenantAcme, DemoTenantGlobex
    async with Medha("tenant_acme", embedder=embedder, settings=settings) as m_acme:
        async with Medha("tenant_globex", embedder=embedder, settings=settings) as m_globex:
            await m_acme.store("How many users?", "SELECT COUNT(*) FROM acme_users")
            await m_globex.store("How many users?", "SELECT COUNT(*) FROM globex_accounts")

            hit_acme   = await m_acme.search("User count")
            hit_globex = await m_globex.search("User count")

            print(f"Acme   → {hit_acme.generated_query}")
            print(f"Globex → {hit_globex.generated_query}")

## Weaviate vs Qdrant — When to Choose Which

| Feature | Weaviate | Qdrant |
|---|---|---|
| **Vector index** | HNSW | HNSW + quantization |
| **Metadata filtering** | GraphQL + structured | JSON payload filters |
| **Multi-tenancy** | Native class-level isolation | Collection-level isolation |
| **Knowledge graph** | Built-in cross-references | Not supported |
| **Quantization** | No | Scalar / Binary |
| **Managed cloud** | Weaviate Cloud | Qdrant Cloud |
| **Best for** | Complex filtering, knowledge graphs | Pure vector, quantization |

**Choose Weaviate** when you need advanced metadata filtering, knowledge-graph
cross-references, or native multi-tenancy at the schema level.

**Choose Qdrant** when you need vector-first performance, quantization, or
a lighter operational footprint.